In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings("ignore")

In [3]:
data = pd.read_csv("patient_data_processed.csv")

X = data.drop(columns=['Patient', 'Stages', 'Systolic', 'Diastolic'])
y = data['Stages']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
models = {

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=9))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=10,
        random_state=42
    ),

    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel='linear'))
    ]),

    "RBF SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel='rbf'))
    ]),

    "Ridge Classifier": Pipeline([
        ("scaler", StandardScaler()),
        ("model", RidgeClassifier())
    ]),

    "Gaussian Naive Bayes": GaussianNB()
}

In [7]:
results = []

for name, model in models.items():

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    cv_scores = cross_val_score(model, X, y, cv=5)
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        cv_mean,
        cv_std
    ])

In [9]:
comparison_df = pd.DataFrame(results, columns=[
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "CV Mean Accuracy",
    "CV Std Dev"
])

comparison_df = comparison_df.sort_values(by="Accuracy", ascending=False)

comparison_df

,Model,Accuracy,Precision,Recall,F1-Score,CV Mean Accuracy,CV Std Dev
5,RBF SVM,0.791781,0.845250,0.791781,0.793472,0.711781,0.070925
0,Logistic Regression,0.786301,0.845468,0.786301,0.784219,0.712877,0.082035
4,Linear SVM,0.786301,0.845468,0.786301,0.784219,0.722740,0.118023
6,Ridge Classifier,0.780822,0.858949,0.780822,0.773168,0.722740,0.073869
1,KNN,0.767123,0.790141,0.767123,0.772776,0.763836,0.107618
2,Random Forest,0.767123,0.794125,0.767123,0.771827,0.714521,0.092559
3,Decision Tree,0.742466,0.750184,0.742466,0.744394,0.693699,0.109279
7,Gaussian Naive Bayes,0.723288,0.852677,0.723288,0.729577,0.723836,0.211534
